In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.metrics import (balanced_accuracy_score, roc_auc_score,
                             classification_report, confusion_matrix)

# ---- load ----
"""CSV = os.path.expanduser('/Users/lauraangelican/Desktop/studentship/datasets/griffin training dataset/dataset_v2 (1).csv')   # <- your Griffin CSV"""
CSV = os.path.expanduser('/Users/ROG/IHE-project/data/processed/dataset_v2.csv')
df = pd.read_csv(CSV)

# drop the one fully-idle/empty trial (no tools active) -- it's not a real attempt
df = df[df["active_tool_count"] > 0].copy()

# ---- label: expert = 1, trainee = 0 ----
df["label"] = (df["role"] == "expert").astype(int)

# ---- choose features ----
# EXCLUDE: ids, the label, and anything that leaks the answer.
# target_score / percentage_score ARE the skill score (and the regression
# target) -- feeding them in would be cheating. Bookkeeping counts are
# dropped too; we keep the actual movement metrics.
exclude = {
    "file_name", "role", "participant_id", "task", "task_clean", "trial",
    "label", "target_score", "percentage_score",
    "n_rows", "n_unique_timestamps", "n_usms", "active_tool_count",
    "all_usms_fully_idle", "USM1_active", "USM2_active", "USM3_active",
}
# also drop per-tool sample-count / alignment-count bookkeeping columns
feature_cols = [c for c in df.columns
                if c not in exclude
                and not c.endswith("_n_samples")
                and not c.endswith("_n_aligned_samples")
                and pd.api.types.is_numeric_dtype(df[c])]

X = df[feature_cols].values
y = df["label"].values
groups = df["participant_id"].values

print(f"{len(df)} trials, {len(feature_cols)} features")
print(f"Participants: {df['participant_id'].nunique()} "
      f"({df.groupby('participant_id')['label'].first().sum()} expert, "
      f"{(df.groupby('participant_id')['label'].first()==0).sum()} trainee)")
print(f"Class balance: {int((y==0).sum())} trainee vs {int((y==1).sum())} expert "
      f"({100*y.mean():.0f}% expert)\n")

# a "dumb" baseline: always guess the majority class (trainee)
print(f"Baseline (always guess trainee) accuracy = {(y==0).mean():.2f}")
print("-> that's why we DON'T judge on plain accuracy here.\n")

# ---- model: impute -> scale -> classifier (balanced for imbalance) ----
def make_model(kind):
    if kind == "logreg":
        clf = LogisticRegression(max_iter=2000, class_weight="balanced")
    else:
        clf = RandomForestClassifier(n_estimators=300, random_state=0,
                                     class_weight="balanced")
    return Pipeline([("impute", SimpleImputer(strategy="median")),
                     ("scale", StandardScaler()),
                     ("clf", clf)])

# ---- subject-wise CV that ALSO keeps class balance across folds ----
# StratifiedGroupKFold: splits by participant (like before) AND keeps the
# expert/trainee ratio steady in each fold -- important when experts are rare.
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)

for kind in ["logreg", "forest"]:
    model = make_model(kind)
    y_pred  = cross_val_predict(model, X, y, groups=groups, cv=cv)
    y_proba = cross_val_predict(model, X, y, groups=groups, cv=cv,
                                method="predict_proba")[:, 1]

    print(f"===== {kind.upper()} =====")
    # balanced accuracy = average of how well we do on EACH class
    print(f"Balanced accuracy: {balanced_accuracy_score(y, y_pred):.2f}")
    print(f"ROC-AUC:           {roc_auc_score(y, y_proba):.2f}")
    print(classification_report(y, y_pred,
          target_names=["trainee", "expert"], zero_division=0))
    print("confusion matrix [rows=true, cols=pred], order [trainee, expert]:")
    print(confusion_matrix(y, y_pred), "\n")

216 trials, 116 features
Participants: 33 (5 expert, 28 trainee)
Class balance: 196 trainee vs 20 expert (9% expert)

Baseline (always guess trainee) accuracy = 0.91
-> that's why we DON'T judge on plain accuracy here.

===== LOGREG =====
Balanced accuracy: 0.89
ROC-AUC:           0.97
              precision    recall  f1-score   support

     trainee       0.98      0.94      0.96       196
      expert       0.59      0.85      0.69        20

    accuracy                           0.93       216
   macro avg       0.79      0.89      0.83       216
weighted avg       0.95      0.93      0.94       216

confusion matrix [rows=true, cols=pred], order [trainee, expert]:
[[184  12]
 [  3  17]] 

===== FOREST =====
Balanced accuracy: 0.77
ROC-AUC:           0.94
              precision    recall  f1-score   support

     trainee       0.96      0.94      0.95       196
      expert       0.50      0.60      0.55        20

    accuracy                           0.91       216
   macro a